# 1.5 Memory

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Learning objectives

- Explain why language models are stateless and why "memory" is just re-sending the conversation with every call.
- Show why a naive ever-growing buffer breaks: context-window limits, rising cost, and the lost-in-the-middle effect.
- State where the legacy memory classes went (`langchain-classic`, maintenance mode) and why new code does not use `ConversationBufferMemory`.
- Wire a chain to per-session memory with `RunnableWithMessageHistory`, using a session id and a `MessagesPlaceholder`.
- Describe the modern memory patterns that replace the legacy classes: windowing, summary memory, and token budgets, and point to where LangGraph checkpointers take over in Module 4.

## The one idea: models do not remember anything

This is the point that saves the most confusion in the whole course, so say it plainly: a chat model is stateless. Every `invoke` starts from zero. If a chatbot "remembers" your name on the second message, it is because the application took the whole conversation so far and pasted it into the new request. There is no memory inside the model. There is memory in your code, in the list of messages you keep and re-send.

Watch it fail, so the lesson lands:

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

model.invoke("My favourite city is Porto.").content          # pleasant reply
model.invoke("What is my favourite city?").content           # "I don't know" (it never saw the first message)

Two separate calls, two separate worlds. The fix is to carry the history yourself:

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

history = [
    HumanMessage(content="My favourite city is Porto."),
    AIMessage(content="Noted. Porto it is."),
    HumanMessage(content="What is my favourite city?"),
]
print(model.invoke(history).content)  # Porto

Memory in LangChain is exactly this: a list of messages, loaded before the call and saved after it. Every memory strategy in this lesson is a different answer to "which messages do we keep, and how do we keep the list small?"

## Why the naive buffer breaks

The simplest memory keeps every message forever and re-sends them all. It works beautifully for ten messages and then breaks in three ways, all at once:

1. **The context window.** Every model has a maximum input size. `gpt-4o-mini` accepts far more than early models, but a long support conversation with tool outputs still hits the ceiling. The API then rejects the call, or (worse, with some providers) silently truncates your input and the model answers from a butchered conversation.
2. **Cost.** Input tokens cost money, and you pay for the whole history on every turn. A conversation that is 20,000 tokens long costs roughly twenty times what the first turn cost, every single reply, forever.
3. **Lost in the middle.** Even when the history fits, long contexts degrade answers. Models attend well to the start and the end of a long input and worse to the middle. Buried instructions, earlier corrections, and user preferences in the middle of a long transcript get ignored. Developers notice this as the model "forgetting" something it clearly saw; it is usually an attention failure, not a memory failure.

So the goal is not to remember everything. The goal is to keep the working set small while preserving what the conversation actually needs. That is what the rest of the lesson is about.

## Where the old memory classes went

If you learned LangChain before 2026, you met `ConversationBufferMemory`, `ConversationSummaryMemory`, and friends. In LangChain 1.0 (October 2025) these moved to the **`langchain-classic`** package, which is in maintenance mode: bug fixes only, no new features, and the docs no longer recommend them for new projects.

Do not build on them in this course. They hide too much (magic prompt rewriting, implicit token counting) and they couple memory to the old chain classes. The modern replacement is explicit: you manage a message list, and you wrap your chain with `RunnableWithMessageHistory`, which handles the load-inject-save cycle. You stay in control of the list, so you can window it, summarise it, or budget it yourself.

## RunnableWithMessageHistory: the current bridge

`RunnableWithMessageHistory` wraps any LCEL chain and gives it per-session memory. Three pieces:

1. A **history factory**: a function that takes a session id and returns that session's message store.
2. A **`MessagesPlaceholder`** in your prompt: the slot where history gets injected.
3. The **session id**, passed in the config at invoke time.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import BaseMessage
from pydantic import BaseModel, Field

load_dotenv()

class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    messages: list[BaseMessage] = Field(default_factory=list)

    def add_messages(self, messages: list[BaseMessage]) -> None:
        self.messages.extend(messages)

    def clear(self) -> None:
        self.messages = []

store: dict[str, InMemoryHistory] = {}

def get_session_history(session_id: str) -> InMemoryHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

chain = prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user_101"}}

chain_with_history.invoke({"question": "My favourite city is Porto."}, config=config)
reply = chain_with_history.invoke({"question": "What is my favourite city?"}, config=config)
print(reply.content)  # Porto

What happened under the hood on the second call: the wrapper took the session id, fetched `user_101`'s messages, placed them into the `history` placeholder, ran the chain on the full message list, then appended the new human message and the AI reply to the store. A different session id (`"user_202"`) would see an empty history: clean isolation between users, no extra code.

The two key arguments deserve a second look:

- `input_messages_key="question"` tells the wrapper which input field holds the new user message (so it can save it to the store).
- `history_messages_key="history"` must match the `MessagesPlaceholder(variable_name="history")` in the prompt (so the loaded history lands in the right slot).

Note this store is a plain dict: it vanishes when the process restarts. For production you swap the factory to return a database-backed history (Redis, Postgres, MongoDB integrations exist) without touching the chain. The chain never knows or cares where messages live.

## Modern patterns: windowing, summary, token budgets

`RunnableWithMessageHistory` handles storage. It does not solve the ever-growing buffer; the list still grows without bound. Three patterns, which you combine:

**Windowing: keep the last N messages.** The simplest fix. Before injecting history, slice it. You can do this in the history factory by returning only the tail, or trim inside a `RunnableLambda` step. Ten to twenty recent messages cover most conversations, and the cost stays flat.

**Summary memory: compress the old, keep the recent.** When the history passes a threshold, ask the model to summarise the older part into a compact paragraph, store that paragraph as a system message, and drop the raw messages it replaced. The user told you their name, their project, and three preferences in message 4; a summary like "User is Saud, building a support bot, prefers concise answers" preserves the facts at a fraction of the tokens. The trade: summaries lose detail. Re-summarise periodically rather than once, and keep the last few turns verbatim so the immediate context stays exact.

**Token budgets: decide in tokens, not messages.** "Last 10 messages" is a rough proxy. Better: estimate the token count of the history (roughly 4 characters per token for English, or use the model's token counter) and trim or summarise until the history fits under a budget you chose, say 4,000 tokens, leaving room for the system prompt, the current question, and the reply. A budget makes the failure mode explicit: the chain never overflows, by construction.

A sketch of windowing wired into the factory, since it is the one you will use first:

In [ ]:
WINDOW = 10  # keep the last 10 messages

def get_windowed_history(session_id: str) -> InMemoryHistory:
    full = get_session_history(session_id)
    windowed = InMemoryHistory()
    windowed.messages = full.messages[-WINDOW:]
    return windowed

The wrapper loads through this factory, so it only ever sees the window. The full history stays in `store` if you need it later (for analytics or re-summarising). This is deliberately simple; a production version trims by tokens and persists the window, but the shape is the same.

## Where LangGraph takes over (Modules 4 and 5)

`RunnableWithMessageHistory` is the bridge for LCEL chains, and it is enough for Module 1's lab. But agents in Module 4 are graphs, not chains, and graphs manage memory differently: a **checkpointer** persists the entire graph state (messages, intermediate values, step counts) after every node, keyed by thread id. The pattern is the same idea one level up: the thread id plays the role of the session id, and the persisted state plays the role of the message list. You will also meet time travel (rewinding a conversation to an earlier checkpoint) and human-in-the-loop, which are checkpoint features, not chain features. Learn the message-list mental model now; LangGraph reuses it.

## A word on conversation logs and GDPR (EU angle)

Memory means you are storing what users said. In the EU that is personal data the moment a user can be identified from it, and the GDPR applies: you need a lawful basis, you should keep only what you need, and users have a right to erasure ("delete my data"). Practical consequences for your agent: scope histories per user and per session so deletion is a key lookup, not a hunt through a shared table; set a retention policy (for example, drop sessions inactive for 90 days) instead of keeping transcripts forever; and keep chat logs out of training pipelines unless the user consented. None of this is a feature of LangChain; it is your application code and your database. Module 6.5 returns to it with checklists.

## What can go wrong

- **Forgetting the `MessagesPlaceholder`.** History loads but has nowhere to go, so the model still forgets. The wrapper does not warn you. If memory "is not working", check the placeholder name against `history_messages_key` first.
- **Mismatched key names.** `input_messages_key="question"` while your prompt expects `{input}` means the new message never reaches the model, or is saved under the wrong key. Keep the three names (prompt variable, input key, placeholder variable) in one place and double-check them together.
- **Forgetting the session id.** `RunnableWithMessageHistory` requires `config={"configurable": {"session_id": ...}}` on every call. Omit it and the wrapper raises; reuse one id for all users and everyone shares one brain.
- **Unbounded growth.** The default pattern appends forever. Without windowing, summarising, or a budget, long conversations overflow the context window or the budget, usually at 2 AM in production. Pick a bounding strategy before the first deploy, not after the first incident.
- **Summaries that lose the point.** A summary that drops the user's correction ("actually use euros, not dollars") makes the agent confidently wrong for the rest of the conversation. Re-summarise from the raw recent turns plus the old summary, and keep corrections verbatim in the window.
- **Session id collisions.** Predictable ids (`"session_1"`, an incrementing counter) let one user stumble into another's history. Use the authenticated user id plus a conversation id as the session key.
- **In-memory stores in production.** The dict-based store in this lesson dies with the process and is not shared between workers. Fine for the lab; a database-backed history is required for anything real.
- **Storing personal data without a plan.** Chat logs accumulate names, emails, and complaints. Without retention and deletion handling you are one subject-access request away from a very bad week. Decide the policy when you design the store.

## Check your understanding

**1. Why does the model "forget" your name between two `invoke` calls, and what is actually happening?**
The model is stateless: each `invoke` starts with no memory of previous calls. Remembering requires your code to keep the message list and re-send it with the next request. Nothing persists inside the model.

**2. Name the three ways a naive ever-growing message buffer breaks.**
It exceeds the model's context window (rejected or truncated calls), it makes every reply cost more (you re-pay for the whole history each turn), and long contexts trigger lost-in-the-middle behaviour where the model ignores details buried mid-transcript.

**3. You copied `ConversationBufferMemory` from a 2024 tutorial and the import fails. What happened and what do you use instead?**
The legacy memory classes moved to `langchain-classic` in LangChain 1.0, which is in maintenance mode. New code does not import them; use `RunnableWithMessageHistory` to wrap an LCEL chain, with a session-id-keyed history factory and a `MessagesPlaceholder` in the prompt.

**4. In `RunnableWithMessageHistory`, what breaks if the prompt has no `MessagesPlaceholder`, and what breaks if you forget the session id in the config?**
Without the placeholder, history loads but is never injected into the prompt, so the model still forgets (silently). Without the session id in the config, the wrapper cannot pick a history and the call fails; using one shared id for everyone merges all users' conversations.

**5. Your agent will store EU users' chat transcripts. Name two GDPR-driven design decisions.**
Scope history per user and session so erasure is a key lookup, and set a retention policy (for example, delete inactive sessions after 90 days) rather than keeping transcripts indefinitely. A lawful basis and keeping logs out of training data without consent complete the picture.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.